# Advanced Mutual Fund Analytics

**Comprehensive analysis including:**
- Historical Value-at-Risk (VaR) and Conditional VaR (CVaR) for 40 schemes
- Rolling 90-day Sharpe ratios with time-series visualization
- Investor cohort analysis by first transaction year
- SIP continuity analysis and at-risk detection
- Risk-based fund recommender system
- Sector HHI concentration metrics
- Strategic insights and recommendations

## 1. Import Libraries and Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Load data
fund_master = pd.read_csv('data/processed/01_fund_master.csv')
nav_history = pd.read_csv('data/processed/02_nav_history.csv')
investor_transactions = pd.read_csv('data/processed/08_investor_transactions.csv')
portfolio_holdings = pd.read_csv('data/processed/09_portfolio_holdings.csv')

# Convert date columns
nav_history['date'] = pd.to_datetime(nav_history['date'])
investor_transactions['transaction_date'] = pd.to_datetime(investor_transactions['transaction_date'])
portfolio_holdings['portfolio_date'] = pd.to_datetime(portfolio_holdings['portfolio_date'])

print(f"Fund Master: {fund_master.shape}")
print(f"NAV History: {nav_history.shape}")
print(f"Investor Transactions: {investor_transactions.shape}")
print(f"Portfolio Holdings: {portfolio_holdings.shape}")
print(f"\nUnique schemes in nav_history: {nav_history['amfi_code'].nunique()}")
print(f"Unique schemes in fund_master: {fund_master['amfi_code'].nunique()}")

## 2. Historical VaR (95%) and CVaR for 40 Schemes

**VaR at 95% confidence** = 5th percentile of daily returns (worst case scenario 5% of the time)  
**CVaR (Expected Shortfall)** = Average return when losses exceed VaR threshold

In [ ]:
# Calculate daily returns for each scheme
daily_returns = nav_history.copy()
daily_returns = daily_returns.sort_values(['amfi_code', 'date'])
daily_returns['daily_return'] = daily_returns.groupby('amfi_code')['nav'].pct_change() * 100

# Calculate VaR (95%) and CVaR for each scheme
var_cvar_results = []

for amfi_code in daily_returns['amfi_code'].unique():
    scheme_returns = daily_returns[daily_returns['amfi_code'] == amfi_code]['daily_return'].dropna()
    
    if len(scheme_returns) > 0:
        # VaR at 95% confidence (5th percentile)
        var_95 = scheme_returns.quantile(0.05)
        
        # CVaR (Expected Shortfall) - mean of returns below VaR
        cvar = scheme_returns[scheme_returns <= var_95].mean()
        
        # Additional metrics
        daily_volatility = scheme_returns.std()
        annual_volatility = daily_volatility * np.sqrt(252)
        mean_daily_return = scheme_returns.mean()
        annual_return = mean_daily_return * 252
        
        var_cvar_results.append({
            'amfi_code': amfi_code,
            'VaR_95pct': var_95,
            'CVaR': cvar,
            'Daily_Volatility': daily_volatility,
            'Annual_Volatility': annual_volatility,
            'Avg_Daily_Return': mean_daily_return,
            'Annual_Return': annual_return,
            'Observations': len(scheme_returns)
        })

var_cvar_df = pd.DataFrame(var_cvar_results)

# Merge with fund master for scheme names
var_cvar_df = var_cvar_df.merge(
    fund_master[['amfi_code', 'scheme_name', 'risk_category']],
    on='amfi_code',
    how='left'
)

# Sort by VaR (highest risk first)
var_cvar_df = var_cvar_df.sort_values('VaR_95pct')

print(f"Schemes analyzed: {len(var_cvar_df)}")
print(f"\nTop 10 Highest Risk (Worst VaR):")
print(var_cvar_df.nlargest(10, 'VaR_95pct')[['amfi_code', 'scheme_name', 'VaR_95pct', 'CVaR', 'Annual_Volatility']])

print(f"\nTop 10 Lowest Risk (Best VaR):")
print(var_cvar_df.nsmallest(10, 'VaR_95pct')[['amfi_code', 'scheme_name', 'VaR_95pct', 'CVaR', 'Annual_Volatility']])

# Save VaR/CVaR report
var_cvar_report = var_cvar_df[['amfi_code', 'scheme_name', 'risk_category', 'VaR_95pct', 'CVaR', 
                                 'Annual_Volatility', 'Annual_Return', 'Observations']].copy()
var_cvar_report.columns = ['AMFI Code', 'Scheme Name', 'Risk Category', 'VaR 95%', 'CVaR', 
                            'Annual Volatility (%)', 'Annual Return (%)', 'Trading Days']
var_cvar_report.to_csv('reports/var_cvar_report.csv', index=False)
print(f"\n✓ var_cvar_report.csv saved")

## 3. Rolling 90-Day Sharpe Ratio and Visualization

**Sharpe Ratio** = (Mean Return / Std Dev) × √252  
Rolling 90-day window captures performance momentum and volatility changes

In [ ]:
# Identify 5 key funds (largest by assets or most popular)
key_funds = fund_master.nlargest(5, 'amfi_code')[['amfi_code', 'scheme_name']].to_dict('records')
key_fund_codes = [f['amfi_code'] for f in key_funds]

# Calculate rolling 90-day Sharpe
rolling_sharpe_data = []

for amfi_code in key_fund_codes:
    scheme_data = daily_returns[daily_returns['amfi_code'] == amfi_code].copy()
    scheme_data = scheme_data.sort_values('date')
    
    # Calculate rolling mean and std
    scheme_data['rolling_mean'] = scheme_data['daily_return'].rolling(window=90).mean()
    scheme_data['rolling_std'] = scheme_data['daily_return'].rolling(window=90).std()
    
    # Calculate rolling Sharpe (annualized)
    scheme_data['rolling_sharpe'] = (scheme_data['rolling_mean'] / scheme_data['rolling_std']) * np.sqrt(252)
    
    rolling_sharpe_data.append(scheme_data[['date', 'amfi_code', 'rolling_sharpe']])

rolling_sharpe_df = pd.concat(rolling_sharpe_data, ignore_index=True)
rolling_sharpe_df = rolling_sharpe_df.dropna()

# Plot rolling Sharpe for 5 key funds
fig, axes = plt.subplots(len(key_fund_codes), 1, figsize=(14, 12))
if len(key_fund_codes) == 1:
    axes = [axes]

for idx, amfi_code in enumerate(key_fund_codes):
    data = rolling_sharpe_df[rolling_sharpe_df['amfi_code'] == amfi_code]
    fund_name = fund_master[fund_master['amfi_code'] == amfi_code]['scheme_name'].values[0]
    
    axes[idx].plot(data['date'], data['rolling_sharpe'], linewidth=2, color='steelblue')
    axes[idx].axhline(y=0, color='red', linestyle='--', alpha=0.5)
    axes[idx].fill_between(data['date'], data['rolling_sharpe'], 0, alpha=0.2)
    axes[idx].set_title(f'Rolling 90-Day Sharpe Ratio: {fund_name[:50]}', fontsize=11, fontweight='bold')
    axes[idx].set_ylabel('Sharpe Ratio')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reports/rolling_sharpe_chart.png', dpi=300, bbox_inches='tight')
print("✓ rolling_sharpe_chart.png saved")
plt.show()

# Summary statistics
print("\nRolling 90-Day Sharpe Ratio Summary:")
summary_sharpe = rolling_sharpe_df.groupby('amfi_code')['rolling_sharpe'].describe()
summary_sharpe['scheme_name'] = summary_sharpe.index.map(
    lambda x: fund_master[fund_master['amfi_code'] == x]['scheme_name'].values[0]
)
print(summary_sharpe[['scheme_name', 'mean', 'min', 'max']])

## 4. Investor Cohort Analysis by First Transaction Year

In [ ]:
# Find first transaction year for each investor
first_transaction = investor_transactions.sort_values('transaction_date').groupby('investor_id').agg({
    'transaction_date': 'first',
    'amfi_code': 'count'  # Count total transactions
}).rename(columns={'transaction_date': 'first_transaction_date', 'amfi_code': 'total_transactions'})

first_transaction['cohort_year'] = first_transaction['first_transaction_date'].dt.year
first_transaction = first_transaction.reset_index()

# Join with transaction data
investor_cohort = investor_transactions.merge(first_transaction[['investor_id', 'cohort_year']], on='investor_id')

# Filter for SIP transactions only
sip_transactions = investor_cohort[investor_cohort['transaction_type'] == 'SIP'].copy()

# Cohort analysis
cohort_analysis = sip_transactions.groupby('cohort_year').agg({
    'investor_id': 'nunique',  # Number of investors
    'amount_inr': ['mean', 'sum'],  # Avg SIP and total invested
    'amfi_code': lambda x: x.value_counts().index[0] if len(x) > 0 else None  # Most popular fund
}).reset_index()

cohort_analysis.columns = ['Cohort Year', 'Investor Count', 'Avg SIP Amount', 'Total Invested', 'Top Fund Code']

# Add fund name
cohort_analysis['Top Fund Name'] = cohort_analysis['Top Fund Code'].map(
    lambda x: fund_master[fund_master['amfi_code'] == x]['scheme_name'].values[0] if x else 'N/A'
)

print("\n" + "="*80)
print("INVESTOR COHORT ANALYSIS BY FIRST TRANSACTION YEAR")
print("="*80)
print(cohort_analysis.to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Investor count by cohort
axes[0].bar(cohort_analysis['Cohort Year'].astype(str), cohort_analysis['Investor Count'], color='steelblue', alpha=0.7)
axes[0].set_title('Number of Investors by Cohort Year', fontweight='bold')
axes[0].set_xlabel('Cohort Year')
axes[0].set_ylabel('Investor Count')
axes[0].grid(True, alpha=0.3, axis='y')

# Total invested by cohort
axes[1].bar(cohort_analysis['Cohort Year'].astype(str), cohort_analysis['Total Invested']/1e6, color='darkgreen', alpha=0.7)
axes[1].set_title('Total Invested by Cohort Year (₹ Crores)', fontweight='bold')
axes[1].set_xlabel('Cohort Year')
axes[1].set_ylabel('Total Invested (₹ Cr)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. SIP Continuity Analysis and At-Risk Detection

For investors with 6+ SIP transactions, analyze gaps between consecutive transactions.  
**At-risk investors** have average gap > 35 days (indicates inconsistent SIP behavior)

In [ ]:
# Filter SIP transactions
sip_only = investor_transactions[investor_transactions['transaction_type'] == 'SIP'].copy()
sip_only = sip_only.sort_values(['investor_id', 'transaction_date'])

# Find investors with 6+ SIP transactions
sip_counts = sip_only.groupby('investor_id').size()
investors_with_6plus = sip_counts[sip_counts >= 6].index

# Calculate gaps for qualifying investors
continuity_analysis = []

for investor in investors_with_6plus:
    investor_sips = sip_only[sip_only['investor_id'] == investor]['transaction_date'].values
    
    # Calculate gaps in days
    gaps = np.diff(pd.to_datetime(investor_sips)) / np.timedelta64(1, 'D')
    
    avg_gap = gaps.mean() if len(gaps) > 0 else 0
    max_gap = gaps.max() if len(gaps) > 0 else 0
    
    # Flag as at-risk if avg gap > 35 days
    at_risk = 'Yes' if avg_gap > 35 else 'No'
    
    continuity_analysis.append({
        'investor_id': investor,
        'sip_count': len(investor_sips),
        'avg_gap_days': avg_gap,
        'max_gap_days': max_gap,
        'at_risk_flag': at_risk
    })

continuity_df = pd.DataFrame(continuity_analysis)

# Statistics
total_qualifying = len(investors_with_6plus)
at_risk_count = len(continuity_df[continuity_df['at_risk_flag'] == 'Yes'])
continuity_rate = ((total_qualifying - at_risk_count) / total_qualifying * 100) if total_qualifying > 0 else 0

print("\n" + "="*80)
print("SIP CONTINUITY ANALYSIS")
print("="*80)
print(f"Total Investors with 6+ SIP transactions: {total_qualifying}")
print(f"At-Risk Investors (avg gap > 35 days): {at_risk_count}")
print(f"SIP Continuity Rate: {continuity_rate:.1f}%")

print(f"\nTop 10 At-Risk Investors:")
at_risk_investors = continuity_df[continuity_df['at_risk_flag'] == 'Yes'].nlargest(10, 'avg_gap_days')
print(at_risk_investors[['investor_id', 'sip_count', 'avg_gap_days', 'max_gap_days']].to_string(index=False))

print(f"\nTop 10 Most Consistent Investors (lowest avg gap):")
consistent = continuity_df.nsmallest(10, 'avg_gap_days')
print(consistent[['investor_id', 'sip_count', 'avg_gap_days', 'max_gap_days']].to_string(index=False))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of avg gaps
axes[0].hist(continuity_df['avg_gap_days'], bins=20, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=35, color='red', linestyle='--', linewidth=2, label='At-Risk Threshold (35 days)')
axes[0].set_title('Distribution of Avg Gap Between SIP Transactions', fontweight='bold')
axes[0].set_xlabel('Average Gap (days)')
axes[0].set_ylabel('Investor Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# At-risk vs consistent
at_risk_pct = [at_risk_count, total_qualifying - at_risk_count]
labels = [f'At-Risk\n({at_risk_count})', f'Consistent\n({total_qualifying - at_risk_count})']
axes[1].pie(at_risk_pct, labels=labels, autopct='%1.1f%%', colors=['#ff6b6b', '#51cf66'], startangle=90)
axes[1].set_title('SIP Investor Risk Profile', fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Fund Recommender System Based on Risk Appetite

In [ ]:
# Create fund recommender function
def recommend_funds(risk_appetite, top_n=3):
    """
    Recommend top N funds based on risk appetite.
    
    Parameters:
    -----------
    risk_appetite : str
        One of 'Low', 'Moderate', 'High'
    top_n : int
        Number of top funds to recommend (default: 3)
    
    Returns:
    --------
    pd.DataFrame
        Recommended funds sorted by Sharpe ratio
    """
    
    # Map risk appetite to fund risk categories
    risk_mapping = {
        'Low': ['Low'],
        'Moderate': ['Low', 'Moderate'],
        'High': ['Moderate', 'High']
    }
    
    if risk_appetite not in risk_mapping:
        return f"Invalid risk appetite. Choose from: {list(risk_mapping.keys())}"
    
    target_categories = risk_mapping[risk_appetite]
    
    # Calculate Sharpe ratio for each fund
    sharpe_data = []
    for amfi_code in var_cvar_df['amfi_code']:
        scheme_info = var_cvar_df[var_cvar_df['amfi_code'] == amfi_code].iloc[0]
        annual_return = scheme_info['Annual_Return']
        annual_vol = scheme_info['Annual_Volatility']
        
        if annual_vol > 0:
            sharpe = annual_return / annual_vol
        else:
            sharpe = 0
        
        sharpe_data.append({
            'amfi_code': amfi_code,
            'scheme_name': scheme_info['scheme_name'],
            'risk_category': scheme_info['risk_category'],
            'annual_return': annual_return,
            'annual_volatility': annual_vol,
            'sharpe_ratio': sharpe
        })
    
    sharpe_df = pd.DataFrame(sharpe_data)
    
    # Filter by risk category
    filtered = sharpe_df[sharpe_df['risk_category'].isin(target_categories)]
    
    # Sort by Sharpe ratio (descending) and get top N
    recommendations = filtered.nlargest(top_n, 'sharpe_ratio')[
        ['amfi_code', 'scheme_name', 'risk_category', 'annual_return', 'annual_volatility', 'sharpe_ratio']
    ]
    
    return recommendations

# Test recommender for all three risk appetites
print("\n" + "="*80)
print("FUND RECOMMENDER - TOP 3 FUNDS BY RISK APPETITE")
print("="*80)

for risk_appetite in ['Low', 'Moderate', 'High']:
    print(f"\n{'─'*80}")
    print(f"RISK APPETITE: {risk_appetite.upper()}")
    print(f"{'─'*80}")
    
    recommendations = recommend_funds(risk_appetite, top_n=3)
    
    # Format output table
    display_df = recommendations.copy()
    display_df.columns = ['AMFI Code', 'Scheme Name', 'Risk', 'Annual Return (%)', 'Volatility (%)', 'Sharpe Ratio']
    display_df['Scheme Name'] = display_df['Scheme Name'].str[:50]
    
    print(display_df.to_string(index=False))

## 7. Sector HHI Concentration Analysis

**Herfindahl-Hirschman Index (HHI)** = Σ(weight_i²)  
- HHI ranges from 0 to 10,000
- **High HHI (> 2500)** = Concentrated portfolio (few dominant stocks)
- **Low HHI (< 1500)** = Diversified portfolio

In [ ]:
# Get latest portfolio data (most recent date)
latest_date = portfolio_holdings['portfolio_date'].max()
latest_holdings = portfolio_holdings[portfolio_holdings['portfolio_date'] == latest_date].copy()

# Calculate HHI (sector concentration)
hhi_results = []

for amfi_code in latest_holdings['amfi_code'].unique():
    fund_holdings = latest_holdings[latest_holdings['amfi_code'] == amfi_code]
    
    # Calculate sector-level weights
    sector_weights = fund_holdings.groupby('sector')['weight_pct'].sum()
    
    # Calculate HHI
    hhi = (sector_weights ** 2).sum()
    
    # Diversification level
    if hhi > 2500:
        diversification = 'Highly Concentrated'
    elif hhi > 1500:
        diversification = 'Moderately Concentrated'
    else:
        diversification = 'Well-Diversified'
    
    # Get fund info
    fund_info = fund_master[fund_master['amfi_code'] == amfi_code].iloc[0]
    
    hhi_results.append({
        'amfi_code': amfi_code,
        'scheme_name': fund_info['scheme_name'],
        'category': fund_info['category'],
        'HHI': hhi,
        'Diversification': diversification,
        'Num_Sectors': len(sector_weights),
        'Top_Sector': sector_weights.idxmax(),
        'Top_Sector_Weight': sector_weights.max()
    })

hhi_df = pd.DataFrame(hhi_results)
hhi_df = hhi_df.sort_values('HHI', ascending=False)

print("\n" + "="*80)
print("SECTOR CONCENTRATION ANALYSIS (HHI)")
print("="*80)
print(hhi_df[['amfi_code', 'scheme_name', 'HHI', 'Diversification', 'Top_Sector']].to_string(index=False))

# Distribution of HHI
print(f"\nHHI Statistics:")
print(f"  Mean HHI: {hhi_df['HHI'].mean():.0f}")
print(f"  Median HHI: {hhi_df['HHI'].median():.0f}")
print(f"  Max HHI (Most Concentrated): {hhi_df['HHI'].max():.0f}")
print(f"  Min HHI (Most Diversified): {hhi_df['HHI'].min():.0f}")

print(f"\nConcentration Distribution:")
print(f"  Highly Concentrated (HHI > 2500): {len(hhi_df[hhi_df['HHI'] > 2500])}")
print(f"  Moderately Concentrated (1500-2500): {len(hhi_df[(hhi_df['HHI'] >= 1500) & (hhi_df['HHI'] <= 2500)])}")
print(f"  Well-Diversified (HHI < 1500): {len(hhi_df[hhi_df['HHI'] < 1500])}")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HHI distribution
axes[0].hist(hhi_df['HHI'], bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].axvline(x=1500, color='orange', linestyle='--', linewidth=2, label='Moderate Threshold')
axes[0].axvline(x=2500, color='red', linestyle='--', linewidth=2, label='High Threshold')
axes[0].set_title('Distribution of HHI Across Equity Funds', fontweight='bold')
axes[0].set_xlabel('HHI Value')
axes[0].set_ylabel('Fund Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Top 10 most concentrated
top_concentrated = hhi_df.nlargest(10, 'HHI')
axes[1].barh(range(len(top_concentrated)), top_concentrated['HHI'], color='steelblue', alpha=0.7)
axes[1].set_yticks(range(len(top_concentrated)))
axes[1].set_yticklabels([name[:30] + '...' if len(name) > 30 else name 
                          for name in top_concentrated['scheme_name']], fontsize=9)
axes[1].set_title('Top 10 Most Concentrated Funds (by HHI)', fontweight='bold')
axes[1].set_xlabel('HHI Value')
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 8. Advanced Strategic Insights

### **Insight 1: Highest Risk Funds & Risk Management**

The funds with the highest historical VaR (95%) represent the riskiest investments based on daily return volatility.

In [ ]:
print("\n" + "="*90)
print("ADVANCED STRATEGIC INSIGHTS")
print("="*90)

# INSIGHT 1: Highest Risk Funds
print("\n📊 INSIGHT 1: HIGHEST RISK FUNDS & TAIL RISK EXPOSURE")
print("-" * 90)
top_risk_funds = var_cvar_df.nlargest(5, 'VaR_95pct')[['scheme_name', 'VaR_95pct', 'CVaR', 'Annual_Volatility']]
print("Top 5 Funds with Highest Daily VaR (95%):\n")
for idx, row in top_risk_funds.iterrows():
    print(f"  • {row['scheme_name'][:50]}")
    print(f"    - VaR (95%): {row['VaR_95pct']:.2f}% (worst 5% daily loss)")
    print(f"    - CVaR (Expected Shortfall): {row['CVaR']:.2f}% (avg loss when exceeding VaR)")
    print(f"    - Annual Volatility: {row['Annual_Volatility']:.2f}%\n")

# INSIGHT 2: Investor Demographics
print("\n👥 INSIGHT 2: INVESTOR COHORT COMPOSITION & GROWTH PATTERNS")
print("-" * 90)
latest_cohort = cohort_analysis.iloc[-1] if len(cohort_analysis) > 0 else None
if latest_cohort is not None:
    recent_cohort_year = int(latest_cohort['Cohort Year'])
    recent_investors = int(latest_cohort['Investor Count'])
    
    print(f"Most Recent Cohort (Year {recent_cohort_year}): {recent_investors} investors")
    print(f"Average SIP Amount: ₹{latest_cohort['Avg SIP Amount']:.0f}")
    print(f"Total Invested: ₹{latest_cohort['Total Invested']/1e6:.1f} Cr")
    print(f"Top Fund: {latest_cohort['Top Fund Name'][:50]}")

total_cohort_investors = cohort_analysis['Investor Count'].sum()
print(f"\nTotal Investors Across All Cohorts: {total_cohort_investors}")
print(f"Cohorts with highest growth:")
growth_cohorts = cohort_analysis.nlargest(3, 'Investor Count')
for _, row in growth_cohorts.iterrows():
    print(f"  • {int(row['Cohort Year'])}: {int(row['Investor Count'])} investors, ₹{row['Total Invested']/1e6:.1f} Cr invested")

# INSIGHT 3: SIP Continuity
print("\n⏳ INSIGHT 3: SIP CONTINUITY & INVESTOR RETENTION RISK")
print("-" * 90)
print(f"Key Metrics for 6+ SIP Transaction Investors:")
print(f"  • Total Qualifying Investors: {total_qualifying}")
print(f"  • Average Gap Between Transactions: {continuity_df['avg_gap_days'].mean():.1f} days")
print(f"  • Median Gap: {continuity_df['avg_gap_days'].median():.1f} days")
print(f"  • At-Risk Investors (avg gap > 35 days): {at_risk_count} ({continuity_rate:.1f}%)")
print(f"  • Most Consistent Investors (avg gap ≤ 35 days): {total_qualifying - at_risk_count}")
print(f"\n  Risk Interpretation:")
print(f"  - Investors with consistent SIP patterns (≤35 day gap) show strong commitment")
print(f"  - At-risk investors may require engagement and retention strategies")
print(f"  - SIP Continuity Rate of {continuity_rate:.1f}% indicates {'strong' if continuity_rate > 70 else 'moderate'} investor discipline")

# INSIGHT 4: Fund Concentration
print("\n🎯 INSIGHT 4: PORTFOLIO CONCENTRATION & DIVERSIFICATION RISK")
print("-" * 90)
high_concentration = hhi_df[hhi_df['HHI'] > 2500]
well_diversified = hhi_df[hhi_df['HHI'] < 1500]
print(f"Portfolio Concentration Analysis:")
print(f"  • Highly Concentrated Funds (HHI > 2500): {len(high_concentration)} funds")
print(f"    (These have concentrated bets on specific sectors/stocks)")
print(f"  • Well-Diversified Funds (HHI < 1500): {len(well_diversified)} funds")
print(f"    (These spread risk across many sectors/stocks)")
print(f"\nMost Concentrated Fund:")
most_conc = hhi_df.iloc[0]
print(f"  • {most_conc['scheme_name'][:50]}")
print(f"    HHI: {most_conc['HHI']:.0f} | Top Sector: {most_conc['Top_Sector']} ({most_conc['Top_Sector_Weight']:.1f}%)")
print(f"\nMost Diversified Fund:")
least_conc = hhi_df.iloc[-1]
print(f"  • {least_conc['scheme_name'][:50]}")
print(f"    HHI: {least_conc['HHI']:.0f} | Sectors: {least_conc['Num_Sectors']} | Top: {least_conc['Top_Sector']} ({least_conc['Top_Sector_Weight']:.1f}%)")

# INSIGHT 5: Recommendations & Strategic Actions
print("\n💡 INSIGHT 5: STRATEGIC RECOMMENDATIONS & ACTION ITEMS")
print("-" * 90)

# Best Sharpe ratio funds
sharpe_data = []
for amfi_code in var_cvar_df['amfi_code']:
    scheme_info = var_cvar_df[var_cvar_df['amfi_code'] == amfi_code].iloc[0]
    annual_return = scheme_info['Annual_Return']
    annual_vol = scheme_info['Annual_Volatility']
    if annual_vol > 0:
        sharpe = annual_return / annual_vol
    else:
        sharpe = 0
    sharpe_data.append({'amfi_code': amfi_code, 'scheme_name': scheme_info['scheme_name'], 'sharpe': sharpe})

sharpe_df = pd.DataFrame(sharpe_data).nlargest(3, 'sharpe')

print("1. BEST RISK-ADJUSTED PERFORMERS (Top 3 by Sharpe Ratio):")
for idx, row in sharpe_df.iterrows():
    print(f"   • {row['scheme_name'][:50]} (Sharpe: {row['sharpe']:.2f})")

print("\n2. INVESTOR ENGAGEMENT PRIORITIES:")
print(f"   • Target {at_risk_count} at-risk SIP investors with continuity communication")
print(f"   • Focus retention efforts on cohorts from {int(cohort_analysis['Cohort Year'].max())} (most recent)")
print(f"   • Analyze top investor demographic: {investor_transactions['age_group'].mode().values[0]}, {investor_transactions['gender'].mode().values[0]}")

print("\n3. PORTFOLIO POSITIONING ADVICE:")
print(f"   • Conservative investors: Choose from {len(well_diversified)} well-diversified funds (HHI < 1500)")
print(f"   • Growth investors: Consider {len(high_concentration)} concentrated funds with sector conviction")

print("\n✓ Analysis Complete - All metrics computed and visualized")
print("="*90)